# Manual replication of `test_parameterization_of_pick_up`

This notebook manually walks through each step of the test, replacing pytest fixtures with explicit setup code.

In [1]:
import os
from copy import deepcopy

import numpy as np

# krrood
from krrood.entity_query_language.backends import ProbabilisticBackend
from krrood.entity_query_language.factories import variable_from, underspecified
from krrood.parametrization.model_registries import DictRegistry
from krrood.parametrization.parameterizer import UnderspecifiedParameters
from probabilistic_model.probabilistic_circuit.rx.helper import fully_factorized

# pycram
from pycram.datastructures.dataclasses import Context
from pycram.motion_executor import simulated_robot
from pycram.robot_plans import *  # PickUpAction, GraspDescription, variable, SequentialPlan, ...

# semantic digital twin
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.robots.abstract_robot import Manipulator
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world_description.connections import OmniDrive

print('Imports OK')

Found these qp solvers: ['qpSWIFT', 'qpalm']


/home/malineni/workingdir/cognitive_robot_abstract_machine/semantic_digital_twin/src/semantic_digital_twin/robots/pr2.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Imports OK


## Step 1 — Build the PR2 world (`pr2_world_setup` fixture)

In [2]:
from semantic_digital_twin.world_description.world_entity import Body
from semantic_digital_twin.world_description.connections import Connection6DoF
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName

def world_with_urdf_factory(urdf_path, robot_semantic_annotation=None, drive_connection_type=None,
                             robot_starting_pose=None):
    urdf_parser = URDFParser.from_file(file_path=urdf_path, path_resolver=None)
    world_with_urdf = urdf_parser.parse()
    if robot_semantic_annotation is not None:
        robot_semantic_annotation.from_world(world_with_urdf)

    with world_with_urdf.modify_world():
        map_body = Body(name=PrefixedName("map"))
        localization_body = Body(name=PrefixedName("odom_combined"))
        map_C_localization = Connection6DoF.create_with_dofs(world_with_urdf, map_body, localization_body)
        world_with_urdf.add_connection(map_C_localization)
        c_root_bf = drive_connection_type.create_with_dofs(
            parent=localization_body, child=world_with_urdf.root, world=world_with_urdf
        )
        world_with_urdf.add_connection(c_root_bf)
        c_root_bf.has_hardware_interface = True
        if robot_starting_pose is not None:
            c_root_bf.origin = robot_starting_pose

    return world_with_urdf

urdf_dir = "package://iai_pr2_description/robots/pr2_with_ft2_cableguide.xacro"
pr2_world_setup = world_with_urdf_factory(urdf_dir, PR2, OmniDrive)
print('PR2 world built')

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr

PR2 world built


## Step 2 — Build the apartment world with milk (`apartment_world_setup` fixture)

In [3]:
# notebook lives at test/pycram_test/ — resources are at pycram/resources/
RESOURCES = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "pycram", "resources"))

apartment_world = URDFParser.from_file(
    os.path.join(RESOURCES, "worlds", "apartment.urdf")
).parse()

milk_world = STLParser(
    os.path.join(RESOURCES, "objects", "milk.stl")
).parse()

apartment_world.merge_world_at_pose(
    milk_world,
    HomogeneousTransformationMatrix.from_xyz_rpy(-1.7, 0, 1.07, yaw=np.pi),
)

print('Apartment world with milk built')

Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name apartment/apartment_root
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name apartment/apartment_root
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name apartment/furnitures_root
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name apartment/furnitures_root
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name apartment/walls
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name apartment/furnitures_root
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name apartment/windows
INFO:world.py::601 add_kinematic_struc

Apartment world with milk built


## Step 3 — Merge into combined PR2 + apartment world (`pr2_apartment_world` fixture)

In [4]:
pr2_copy = deepcopy(pr2_world_setup)
PR2.from_world(pr2_copy)  # semantic annotations are lost on copy

apartment_copy = deepcopy(apartment_world)
pr2_copy.merge_world(apartment_copy)

pr2_copy.get_body_by_name("base_footprint").parent_connection.origin = (
    HomogeneousTransformationMatrix.from_xyz_rpy(1.3, 2, 0)
)

pr2_apartment_world = pr2_copy
print('Combined world ready')

INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_bellow_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_laser_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fl_caster_rotation_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fl_caster_l_wheel_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fl_caster_r_wheel_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fr_caster_rotation_link
INFO:wor

Combined world ready


## Step 4 — Create mutable world + robot view + context (`mutable_model_world` fixture)

In [5]:
world = deepcopy(pr2_apartment_world)
robot_view = PR2.from_world(world)
context = Context(world, robot_view)

# print('world:', world)
# print('robot_view:', robot_view)
# print('context:', context)

INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_bellow_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_laser_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fl_caster_rotation_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fl_caster_l_wheel_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fl_caster_r_wheel_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/fr_caster_rotation_link
INFO:wor

## Visualize in RViz2 (optional)

Skip this section if ROS2 is not running. 

RViz2 setup:
- **Fixed Frame**: `map`
- Add **TF** display → topic `/tf`
- Add **MarkerArray** display → topic `/semworld/viz_marker` (QoS Durability = Transient Local)

In [6]:
import threading
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('testing_semworld')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()
print('ROS2 node started')

ROS2 node started


In [8]:
_tf_publisher  = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print('ROS2 publishers started')
print('  Fixed Frame : map')
print('  TF topic    : /tf')
print('  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)')

ROS2 publishers started
  Fixed Frame : map
  TF topic    : /tf
  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)


## Step 5 — Get milk body and create the underspecified pick-up description

In [9]:
milk = world.get_body_by_name("milk.stl")
print('milk body:', milk)

milk_variable = variable_from([milk])

pick_up_description = underspecified(PickUpAction)(
    object_designator=milk_variable,
    arm=...,
    grasp_description=underspecified(GraspDescription)(
        approach_direction=...,
        vertical_alignment=...,
        rotate_gripper=...,
        manipulation_offset=0.05,
        manipulator=variable(Manipulator, world.semantic_annotations),
    ),
)
print('pick_up_description created')

milk body: Body(name=PrefixedName('None/milk.stl'), id=UUID('7aa849b1-8a25-4744-801a-324105bedf51'), index=219)
pick_up_description created


In [35]:
pick_up_description

Match(PickUpAction)

## Step 6 — Resolve and inspect `UnderspecifiedParameters`

In [15]:
pick_up_description.resolve()

Match(PickUpAction)

In [16]:
parameters = UnderspecifiedParameters(pick_up_description)
print('Number of variables:', len(parameters.variables))  # expect 7
print('Variables:', list(parameters.variables.keys()))

Number of variables: 7
Variables: ['PickUpAction.object_designator', 'PickUpAction.arm', 'PickUpAction.grasp_description.approach_direction', 'PickUpAction.grasp_description.vertical_alignment', 'PickUpAction.grasp_description.rotate_gripper', 'PickUpAction.grasp_description.manipulation_offset', 'PickUpAction.grasp_description.manipulator']


In [19]:
list(parameters.variables.values())[-2]

Continuous(PickUpAction.grasp_description.manipulation_offset)

In [17]:
[manipulator_offset] = [
    v for v in parameters.variables.values()
    if v.name.endswith("manipulation_offset")
]

print('manipulator_offset variable:', manipulator_offset)
print('conditioning value:', parameters.assignments_for_conditioning[manipulator_offset])  # expect 0.05

manipulator_offset variable: Continuous(PickUpAction.grasp_description.manipulation_offset, (-inf, inf))
conditioning value: 0.05


## Step 7 — Build the probabilistic model and evaluate

In [20]:
model = fully_factorized(parameters.variables.values())

In [26]:
model

ProbabilisticCircuit with 8 nodes and 7 edges

In [27]:
registry = DictRegistry({PickUpAction: model})

In [31]:
registry.models.keys()

dict_keys([<class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>])

In [32]:
pm_backend = ProbabilisticBackend(registry, 10)
action = next(pm_backend.evaluate(pick_up_description))

print('Sampled action:', action)

Sampled action: PickUpAction(plan_node=None, object_designator=Body(name=PrefixedName('None/milk.stl'), id=UUID('7aa849b1-8a25-4744-801a-324105bedf51'), index=219), arm=LEFT, grasp_description=GraspDescription(approach_direction=<ApproachDirection.LEFT: (<AxisIdentifier.Y: (0, 1, 0)>, 1)>, vertical_alignment=<VerticalAlignment.BOTTOM: (<AxisIdentifier.Z: (0, 0, 1)>, 1)>, manipulator=ParallelGripper(name=PrefixedName('pr2/left_gripper'), id=UUID('16aaf874-88d3-4030-82d4-7c0ebd6d6ae5'), root=Body(name=PrefixedName('pr2/l_gripper_palm_link'), id=UUID('da33d0f5-7d67-4c15-8b7b-44e97408fe0a'), index=84), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('6c574663-059a-41c5-a55c-7b79273b9962'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('c66c16f4-7624-41a3-bcb9-0ca8b90b64fc'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('23780542-9a72-4709-9fbe-861538c07586'), index=22), manipulator=None, sensors=[Camera(name=Prefi

## Step 8 — Build the plan and perform it in the simulated robot

In [33]:
plan = SequentialPlan(context, action)

with simulated_robot:
    try:
        plan.perform()
        print('Plan performed successfully')
    except TimeoutError:
        print('TimeoutError — expected, test passes anyway')

INFO:language.py::281 perform Executing SequentialNode
INFO:base.py::28 perform Performing action PickUpAction
INFO:language.py::281 perform Executing SequentialNode
INFO:base.py::28 perform Performing action ReachAction
INFO:language.py::281 perform Executing SequentialNode
ERROR:motion_executor.py::99 _execute_for_simulation Failed Nodes: [Sequence#0, MoveTCP#2]


TimeoutError — expected, test passes anyway


---
## Shutdown ROS2 Node

In [ ]:
try:
    _ros_node.destroy_node()
    rclpy.shutdown()
    print('ROS2 node stopped')
except Exception:
    print('(ROS2 node was not running)')